In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

In [3]:
file_path = '/content/drive/MyDrive/Retail Analytics System/data/raw/online_retail_raw.csv'

df = pd.read_csv(file_path)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
print("Rows and Columns:", df.shape)
df.info()

Rows and Columns: (1067371, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [5]:
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0


In [6]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 34335


In [7]:
df = df.dropna(subset=['Customer ID'])

print(df.shape)

(824364, 8)


In [8]:
df = df.drop_duplicates()

print(df.shape)

(797885, 8)


In [9]:
df['Invoice'].astype(str).str.startswith('C').sum()

np.int64(18390)

In [10]:
df = df[~df['Invoice'].astype(str).str.startswith('C')]

print(df.shape)

(779495, 8)


In [11]:
(df['Quantity'] <= 0).sum()

np.int64(0)

In [12]:
df = df[df['Quantity'] > 0]

print(df.shape)

(779495, 8)


In [13]:
(df['Price'] <= 0).sum()

np.int64(70)

In [14]:
df = df[df['Price'] > 0]

print(df.shape)

(779425, 8)


In [15]:
df['Revenue'] = df['Quantity'] * df['Price']

df[['Quantity','Price','Revenue']].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [16]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      779425 non-null  object        
 1   StockCode    779425 non-null  object        
 2   Description  779425 non-null  object        
 3   Quantity     779425 non-null  int64         
 4   InvoiceDate  779425 non-null  datetime64[ns]
 5   Price        779425 non-null  float64       
 6   Customer ID  779425 non-null  float64       
 7   Country      779425 non-null  object        
 8   Revenue      779425 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 59.5+ MB


In [17]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day

df[['InvoiceDate','Year','Month','Day']].head()

,InvoiceDate,Year,Month,Day
0,2009-12-01 07:45:00,2009,12,1
1,2009-12-01 07:45:00,2009,12,1
2,2009-12-01 07:45:00,2009,12,1
3,2009-12-01 07:45:00,2009,12,1
4,2009-12-01 07:45:00,2009,12,1


In [18]:
print("Final Shape:", df.shape)

print("\nMissing Values")
print(df.isnull().sum())

Final Shape: (779425, 12)

Missing Values
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
Year           0
Month          0
Day            0
dtype: int64


In [19]:
from google.colab import files

df.to_csv('clean_retail.csv', index=False)

In [20]:
# Fix column names and data types before exporting CSVs

# Customers table
customers = (
    df[['Customer ID', 'Country']]
    .drop_duplicates(subset=['Customer ID'])
    .copy()
)
customers = customers.rename(columns={
    'Customer ID': 'customer_id',
    'Country': 'country'
})
customers['customer_id'] = customers['customer_id'].astype(int)

# Products table
products = (
    df[['StockCode', 'Description']]
    .drop_duplicates(subset=['StockCode'])
    .copy()
)
products = products.rename(columns={
    'StockCode': 'stock_code',
    'Description': 'description'
})

# Orders table
orders = df[['Invoice', 'InvoiceDate', 'Customer ID']].copy()

orders = orders.rename(columns={
    'Invoice': 'invoice_no',
    'InvoiceDate': 'invoice_date',
    'Customer ID': 'customer_id'
})

orders['customer_id'] = orders['customer_id'].astype(float).astype(int)

# one row per invoice only
orders = orders.drop_duplicates(subset=['invoice_no'])

print(orders.head())
print(orders.shape)
print(orders.duplicated(subset=['invoice_no']).sum())
# Sales table
sales = df[['Invoice', 'StockCode', 'Quantity', 'Price', 'Revenue']].copy()

sales = sales.rename(columns={
    'Invoice': 'invoice_no',
    'StockCode': 'stock_code',
    'Quantity': 'quantity',
    'Price': 'price',
    'Revenue': 'revenue'
})

   invoice_no        invoice_date  customer_id
0      489434 2009-12-01 07:45:00        13085
8      489435 2009-12-01 07:46:00        13085
12     489436 2009-12-01 09:06:00        13078
31     489437 2009-12-01 09:08:00        15362
54     489438 2009-12-01 09:24:00        18102
(36969, 3)
0


In [21]:
print(customers.head())
print(products.head())
print(orders.head())
print(sales.head())

    customer_id         country
0         13085  United Kingdom
12        13078  United Kingdom
31        15362  United Kingdom
54        18102  United Kingdom
71        12682          France
  stock_code                          description
0      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS
1     79323P                   PINK CHERRY LIGHTS
2     79323W                  WHITE CHERRY LIGHTS
3      22041         RECORD FRAME 7" SINGLE SIZE 
4      21232       STRAWBERRY CERAMIC TRINKET BOX
   invoice_no        invoice_date  customer_id
0      489434 2009-12-01 07:45:00        13085
8      489435 2009-12-01 07:46:00        13085
12     489436 2009-12-01 09:06:00        13078
31     489437 2009-12-01 09:08:00        15362
54     489438 2009-12-01 09:24:00        18102
  invoice_no stock_code  quantity  price  revenue
0     489434      85048        12   6.95     83.4
1     489434     79323P        12   6.75     81.0
2     489434     79323W        12   6.75     81.0
3     489434      22041   

In [22]:
customers.to_csv('customers.csv', index=False)
products.to_csv('products.csv', index=False)
orders.to_csv('orders.csv', index=False)
sales.to_csv('sales.csv', index=False)

In [23]:
from google.colab import files

files.download('customers.csv')
files.download('products.csv')
files.download('orders.csv')
files.download('sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
print(customers.columns)
print(products.columns)
print(orders.columns)
print(sales.columns)

Index(['customer_id', 'country'], dtype='object')
Index(['stock_code', 'description'], dtype='object')
Index(['invoice_no', 'invoice_date', 'customer_id'], dtype='object')
Index(['invoice_no', 'stock_code', 'quantity', 'price', 'revenue'], dtype='object')
